#Practice with Delta Tables

## List files under the 'learning' Volumen - 3 ways
Note: Let´s see 3 different ways to list files in a path

###Alt1 - %fs Magic Command

In [0]:
%fs ls '/Volumes/workspace/default/learning'

###Alt2 - PySpark

In [0]:
# Using Python code instead
display(dbutils.fs.ls('/Volumes/workspace/default/learning'))
# OBS! dbutils.fs.ls() devuelve un data frame. Por eso se puede usar display()

###Alt3 - Spark SQL

In [0]:
%sql
-- Using SQL stament instead
LIST '/Volumes/workspace/default/learning'

##SQL: Query a CSV data file 

In [0]:
%sql
select * from csv.`/Volumes/workspace/default/learning/people.csv`;
-- Note the bad output due to the CSV file having a non well-defined schema

##PySpark: Create a Delta Table from the CSV file

In [0]:
# Step 1: Create a DataFrame from the CSV file
df = spark.read.csv("/Volumes/workspace/default/learning/people.csv", header=True, inferSchema=True)
display(df)

# Step 2: Create a Delta Table from the DataFrame
df.write.format("delta").saveAsTable("workspace.default.people_table")

##SQL: INSERT data

In [0]:
%sql
insert into people_table
values 
  (11, 'Carlos', 42, 50000),
  (12, 'Lenka', 40, 60000);

  update people_table
  set salary = 70000
  where id = 1;


In [0]:
%sql
select * from people_table order by id;


##SQL: View the table history

In [0]:
%sql
describe history people_table;

##SQL: CTAS statement (Create Table As Select)

###Pre-step: SQL queries to read the CSV file
OBS! Note the different outputs depending on the function used

In [0]:
%sql
-- Alt1 - Use of read_files() function
select * from read_files(
  '/Volumes/workspace/default/learning/customer.csv', -- Si hubiera más CSV files dentro de learning, se leen todos
  format => 'csv'
) limit 10;

-- OBS! Good parsing of the CSV file when using read_files() + adding the _rescued_data column !!!

-- OBS! Notice the column '_rescued_data' which is automatically included to capture any data that does not match the infered schema

In [0]:
%sql
-- Alt2 - Use of SELECT * FROM file_format.`...`
SELECT * FROM csv.`/Volumes/workspace/default/learning/customer.csv`; 

-- OBS! Poor parsing of the file

###CTAS statement

In [0]:
%sql
CREATE OR REPLACE TABLE customers
AS
SELECT * FROM read_files(
  '/Volumes/workspace/default/learning/customer.csv',
  format => 'csv',
  header => true,
  inferSchema => true);

  /*
  -- Bonus: Create the Delta Table explictly indicating the schema
create table customers
as
select *
from read_files(
  '/Volumes/workspace/default/learning/customer.csv',
  format => 'csv',
  header => true,
  schema => '''
    customer_id int,
    first_name string,
    last_name string,
    date_of_birth date,
    gender string''',
  rescueddatacolumn => '_rescued_data'
);
*/

In [0]:
%sql
SELECT * FROM customers;

##SQL: Handle data from the _rescued_data column
_rescued_data stores either NULLS or JSON formatted strings with the values that were not properly parsed

In [0]:
%sql
-- Step 1: Query the table
SELECT * FROM customers_ctas; --Note that customer_ctas_schema has non NULLS  in _rescued_data

In [0]:
%sql
-- Step 2: Access the values from the _rescued_data column - JSON handling
select
  to_date( _rescued_data:date_of_birth, 'dd-MM-yyyy') as date_of_birth
from customer_ctas_schema
limit 10;

In [0]:
%sql
-- Step 3: Update the table accordingly
UPDATE customer_ctas_schema
SET date_of_birth = to_date(_rescued_data:date_of_birth, 'dd-MM-yyyy')
WHERE date_of_birth IS NULL AND _rescued_data:date_of_birth IS NOT NULL;

In [0]:
%sql
-- Step 4: Post-check query
select * from customer_ctas_schema;

##PySpark: Create a Delta Table from a data frame

In [0]:
# Step 1: Ingestion: Create a DataFrame from the CSV file
df = spark.read.csv("/Volumes/workspace/default/learning/customer.csv", header=True, inferSchema=True)
"""""
Alternative declaration
df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .load("/Volumes/workspace/default/learning/customer.csv")
    .withColumn("processing_time", current_timestamp())
    .withColumn("file_name", col("_metadata.file_path"))
)
"""
display(df)
# OBS! Creating the DataFrame in Python does not create by default the _rescued_data column, unless explicitely indicated as one of the read options

# Step 2: Create a Delta Table from the DataFrame
df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.customer_table")

In [0]:
%sql
DESCRIBE TABLE EXTENDED workspace.default.customer_table;

In [0]:
%sql
DESCRIBE HISTORY workspace.default.customer_table;

##PySpark: Read a Delta Table

In [0]:
# Read the table into a data frane and display it
df  = spark.read.table("workspace.default.customer_table")
display(df)
#df.display()

##SQL: COPY INTO statement (Legacy)
Legacy statement to do incremental batch ingestions. 
Recommendation is to use Auto Loader instead

In [0]:
%sql
-- Create empty table with schema
create table if not exists customers_copyinto(
  customer_id int,
  first_name string,
  last_name string,
  date_of_birth date,
  gender string);

-- Populate from a file using copy into
copy into customers_copyinto
from '/Volumes/workspace/default/learning/customer.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true');
-- COPY_OPTIONS can be either mergeSchema (=> schema evolution) or force (=> schema override)
-- COPY_OPTIONS ('mergeSchema' = 'true');

  -- OBS! Lo interesante de este método es que permite hacer incremental loads,
  --      simplemente anadiendo nuevos ficheros al volumen
  --      y ejecutando el mismo copy into.e
  -- En este caso, from 'path del volumen (sin indicar fichero)'

#Medallion (Multi-hop) Architecture
Practice data transformations in SQL

##Bronze layer - Batch ingestion using COPY INTO (Legacy)

In [0]:
%sql
create table if not exists employees_bronze (
  id int,
  name string,
  country string,
  role string);

copy into employees_bronze
from '/Volumes/workspace/default/learning/employees_dataset/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')

##Bronze layer - Using CTAS

In [0]:
%sql
SELECT * FROM csv.`/Volumes/workspace/default/learning/employees_dataset/`;
-- OBS! No se parsea bien el CSV file al ser un formato que no tiene un "well-defined schema", como sí lo tiene JSON o parquet
-- Está bien para ver los datos, pero no crees la tabla usando esta query porque no se va a salir bien.
-- Usar alguna alternativa que permita pasar opciones. COPY INTO, como he hecho arriba functiona, pero es legacy.
-- Vamos a ver otra: Usamos una temp view

In [0]:
%sql
SELECT * FROM text.`/Volumes/workspace/default/learning/employees_dataset`
-- Confirmo que el delimiter is ,

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW temp_view (id INT, name STRING, country STRING, role STRING)
USING csv
OPTIONS (
  path = '/Volumes/workspace/default/learning/employees_dataset/',
  header = 'true',
  delimiter = ','
  );

In [0]:
%sql
SELECT * FROM temp_view;

In [0]:
%sql
-- CTAS
CREATE OR REPLACE TABLE employees_bronze
AS SELECT * FROM temp_view;

In [0]:
%sql
DESCRIBE TABLE EXTENDED employees_bronze;

##Bronze layer - SQL: Enrich table with some metadata for traceability

In [0]:
%sql
CREATE OR REPLACE TABLE employees_bronze
AS
SELECT
  *,
  _metadata.file_path as file_path,
  _metadata.file_modification_time as file_modification_time,
  current_timestamp() as current_timestamp
FROM employees_bronze;

In [0]:
%sql
SELECT * FROM employees_bronze;

##Silver layer - Using CTAS
Basic transformations:
- Select id, name and country from bronze layer
- Convert role to uppercase
- Add 2 new cols: current_timestamp and current_date 

In [0]:
%sql
create table if not exists employees_silver as
select
  id,
  name,
  country,
  upper(role) as role,
  current_timestamp() as current_timestamp,
  date(current_timestamp) as current_date
from employees_bronze;

In [0]:
%sql
select * from employees_silver;

##Gold layer - Create VIEW
Aggregate the silver table to create the gold table.
Steps:
- Create a temp_view that aggregates the total number of employees by role
-  Create a table total_roles_gold (Data Mart)

In [0]:
%sql
-- No es muy elegante esta cell
-- Step 1
create view temp_view as
select
  role,
  count(*) as total
from employees_silver
group by role;
  
-- Step 2
create table if not exists total_roles_gold (
  role string,
  total int
);

insert into total_roles_gold
select * from temp_view;

select * from total_roles_gold;